In [7]:
import ee

In [8]:
ee.Initialize(project='eecc-maureen')

#### Step 1. Data Processing and Export with GEE

In [30]:
# Porto Alegre bounding box [minLon, minLat, maxLon, maxLat]
poa = ee.Geometry.Rectangle([-51.30344, -30.26945, -51.018852, -29.932474])

# --- Land cover: Dynamic World (annual composite, 9 classes) ---
landcover = (ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
    .filterBounds(poa)
    .filterDate("2025-01-01", "2026-01-01")
    .select("label")
    .mode())  # (per-pixel most frequent value over time)

# Reproject to Web Mercator so tiling is straightforward
img = (landcover
       .clip(poa)
       .reproject(crs="EPSG:3857")
       .toInt16())

task = ee.batch.Export.image.toDrive(
    image=img,
    description="poa_dynamicworld_10m_3857_v1_2025",
    folder="gee_exports",                   
    fileNamePrefix="poa_dynamicworld_10m_3857_v1_2025",
    region=poa,
    scale=10,
    crs="EPSG:3857",
    maxPixels=1e13,
    fileFormat="GeoTIFF"
)

task.start()
print("Started Drive export:", task.id)


Started Drive export: KEUO7NM65CESYKY5T444LHBC


In [11]:
# import time

# def wait_for_task(task):
#     """Poll task status until complete or failed."""
#     while task.active():
#         status = task.status()
#         state = status.get('state', 'UNKNOWN')
#         print(f"  Status: {state}")
#         time.sleep(10)
#     status = task.status()
#     if status.get('state') == 'COMPLETED':
#         print("Export finished successfully.")
#     else:
#         print("Export ended with status:", status)

# wait_for_task(task)

#### Step 2. Convert to GOC and Generate Tiles

In [31]:
# Paths (adjust if your data lives elsewhere)
from pathlib import Path
import subprocess

base = Path("data")
in_tif = base / "poa_dynamicworld_10m_3857_v1_2025.tif"
out_dir = Path("out/poa2025")
cog_tif = out_dir / "poa_dynamicworld_10m_3857_v1_2025_cog.tif"
tiles_dir = out_dir / "tiles_visual"

out_dir.mkdir(parents=True, exist_ok=True)

In [32]:
# Step 3: Convert to COG (with overviews in one step - avoids COG layout issues)
subprocess.run([
    "gdal_translate", str(in_tif), str(cog_tif),
    "-of", "COG",
    "-ot", "Byte",
    "-co", "COMPRESS=DEFLATE",
    "-co", "RESAMPLING=NEAREST",
    "-co", "OVERVIEWS=AUTO"
], check=True)
print("Created COG with overviews:", cog_tif)

Input file size is 3169, 4338
0...10...20...30...40...50...60...70...80...90...100 - done.
Created COG with overviews: out/poa2025/poa_dynamicworld_10m_3857_v1_2025_cog.tif


In [33]:
# Colorize: map class IDs 0-8 to Dynamic World colors (fixes black tiles)
import shutil

colorized_tif = out_dir / "poa_dynamicworld_colorized.tif"
colors_txt = base / "dynamic_world_colors.txt"
subprocess.run([
    "gdaldem", "color-relief", str(cog_tif), str(colors_txt), str(colorized_tif)
], check=True)

# Preflight: ensure gdal2tiles exists and its Python runtime has numpy
gdal2tiles = shutil.which("gdal2tiles.py")
if not gdal2tiles:
    raise RuntimeError("gdal2tiles.py not found in PATH. Install GDAL first (e.g., `brew install gdal`).")

gdal2tiles_python = None
with open(gdal2tiles, "r", encoding="utf-8", errors="ignore") as f:
    first_line = f.readline().strip()

if first_line.startswith("#!"):
    shebang_parts = first_line[2:].split()
    if shebang_parts:
        if shebang_parts[0].endswith("env") and len(shebang_parts) > 1:
            gdal2tiles_python = shutil.which(shebang_parts[1])
        else:
            gdal2tiles_python = shebang_parts[0]

if not gdal2tiles_python:
    gdal2tiles_python = shutil.which("python3") or shutil.which("python")

if not gdal2tiles_python:
    raise RuntimeError("Could not determine Python interpreter for gdal2tiles.py")

try:
    subprocess.run([gdal2tiles_python, "-c", "import numpy"], check=True, capture_output=True)
except subprocess.CalledProcessError as e:
    raise RuntimeError(
        "Missing numpy in gdal2tiles runtime. Install with: "
        f"`{gdal2tiles_python} -m pip install numpy`"
    ) from e

# Generate XYZ tiles from colorized RGB
tiles_dir.mkdir(parents=True, exist_ok=True)
subprocess.run([
    "gdal2tiles.py",
    "-r", "near",
    "-z", "8-15",
    "--xyz",
    "-w", "none",
    str(colorized_tif),
    str(tiles_dir)
], check=True)

0...10...20...30...40...50...60...70...80...90...100 - done.


Generating Base Tiles:


0...10...20...30...40...50...60...70...80...90...100 - done in 00:00:05.
0

Generating Overview Tiles:


...10...20...30...40...50...60...70...80...90...100 - done.


CompletedProcess(args=['gdal2tiles.py', '-r', 'near', '-z', '8-15', '--xyz', '-w', 'none', 'out/poa2025/poa_dynamicworld_colorized.tif', 'out/poa2025/tiles_visual'], returncode=0)

#### Export with values 

In [34]:
# Step 3c: Generate VALUE tiles (R=value, G=0, B=0) for client-side hover lookup
# Same extent/resolution as styled tiles - browser reads R channel to get class ID
value_encoded_tif = out_dir / "poa_dynamicworld_value_encoded.tif"
value_colors_txt = base / "value_encoding_colors.txt"
value_tiles_dir = out_dir / "tiles_values"

subprocess.run([
    "gdaldem", "color-relief", str(cog_tif), str(value_colors_txt), str(value_encoded_tif)
], check=True)

value_tiles_dir.mkdir(parents=True, exist_ok=True)
subprocess.run([
    "gdal2tiles.py",
    "-r", "near",
    "-z", "8-15",
    "--xyz",
    "-w", "none",
    str(value_encoded_tif),
    str(value_tiles_dir)
], check=True)
print("Value tiles written to:", value_tiles_dir)

0...10...20...30...40...50...60...70...80...90...100 - done.


Generating Base Tiles:


0...10...20...30...40...50...60...70...80...90...100 - done.
0.

Generating Overview Tiles:


..10...20...30...40...50...60...70...80...90...100 - done.
Value tiles written to: out/poa2025/tiles_values
